In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("Streaming Process Files")
    .config("spark.streaming.stopGracefullyOnShutdown", True)
    .master("local[2]")
    .getOrCreate()
)
spark

In [2]:
# To allow automatic shcemaInference while reading
# spark.conf.set("spark.sql.streaming.schemaInference", True)


#Create the streaming_df to read form input directory

streaming_df = (
    spark.read
    .format("json")
    .load("data/input/device_files/")
)



In [3]:
# To the schema of the data, place a sample json file and change readStream to read
streaming_df.printSchema()

root
 |-- customerId: string (nullable = true)
 |-- data: struct (nullable = true)
 |    |-- devices: array (nullable = true)
 |    |    |-- element: struct (containsNull = true)
 |    |    |    |-- deviceId: string (nullable = true)
 |    |    |    |-- measure: string (nullable = true)
 |    |    |    |-- status: string (nullable = true)
 |    |    |    |-- temperature: long (nullable = true)
 |-- eventId: string (nullable = true)
 |-- eventOffset: long (nullable = true)
 |-- eventPublisher: string (nullable = true)
 |-- eventTime: string (nullable = true)



In [4]:
# lets explode the data as devices contains list/array of device reading

from pyspark.sql.functions import explode

exploded_df = streaming_df.withColumn("data_devices", explode("data.devices")).drop("data")

In [5]:
exploded_df.printSchema()

root
 |-- customerId: string (nullable = true)
 |-- eventId: string (nullable = true)
 |-- eventOffset: long (nullable = true)
 |-- eventPublisher: string (nullable = true)
 |-- eventTime: string (nullable = true)
 |-- data_devices: struct (nullable = true)
 |    |-- deviceId: string (nullable = true)
 |    |-- measure: string (nullable = true)
 |    |-- status: string (nullable = true)
 |    |-- temperature: long (nullable = true)



In [6]:
exploded_df.show()

+----------+--------------------+-----------+--------------+--------------------+--------------------+
|customerId|             eventId|eventOffset|eventPublisher|           eventTime|        data_devices|
+----------+--------------------+-----------+--------------+--------------------+--------------------+
|   CI00103|e3cb26d3-41b2-49a...|      10001|        device|2023-01-05 11:13:...|{D001, C, ERROR, 15}|
|   CI00103|e3cb26d3-41b2-49a...|      10001|        device|2023-01-05 11:13:...|{D002, C, SUCCESS...|
+----------+--------------------+-----------+--------------+--------------------+--------------------+



In [7]:
exploded_df.show(truncate=False)

+----------+------------------------------------+-----------+--------------+--------------------------+----------------------+
|customerId|eventId                             |eventOffset|eventPublisher|eventTime                 |data_devices          |
+----------+------------------------------------+-----------+--------------+--------------------------+----------------------+
|CI00103   |e3cb26d3-41b2-49a2-84f3-0156ed8d7502|10001      |device        |2023-01-05 11:13:53.643364|{D001, C, ERROR, 15}  |
|CI00103   |e3cb26d3-41b2-49a2-84f3-0156ed8d7502|10001      |device        |2023-01-05 11:13:53.643364|{D002, C, SUCCESS, 16}|
+----------+------------------------------------+-----------+--------------+--------------------------+----------------------+



In [8]:
# Flatten the exploded df
from pyspark.sql.functions import col
flattened_df = (
    exploded_df
    .withColumn("deviceID", col("data_devices.deviceId"))
    .withColumn("measure", col("data_devices.measure"))
    .withColumn("status", col("data_devices.status"))
    .withColumn("temperature", col("data_devices.temperature"))
    .drop("data_devices")
)

In [9]:
flattened_df.printSchema()

root
 |-- customerId: string (nullable = true)
 |-- eventId: string (nullable = true)
 |-- eventOffset: long (nullable = true)
 |-- eventPublisher: string (nullable = true)
 |-- eventTime: string (nullable = true)
 |-- deviceID: string (nullable = true)
 |-- measure: string (nullable = true)
 |-- status: string (nullable = true)
 |-- temperature: long (nullable = true)



In [10]:
flattened_df.show(truncate=False)

+----------+------------------------------------+-----------+--------------+--------------------------+--------+-------+-------+-----------+
|customerId|eventId                             |eventOffset|eventPublisher|eventTime                 |deviceID|measure|status |temperature|
+----------+------------------------------------+-----------+--------------+--------------------------+--------+-------+-------+-----------+
|CI00103   |e3cb26d3-41b2-49a2-84f3-0156ed8d7502|10001      |device        |2023-01-05 11:13:53.643364|D001    |C      |ERROR  |15         |
|CI00103   |e3cb26d3-41b2-49a2-84f3-0156ed8d7502|10001      |device        |2023-01-05 11:13:53.643364|D002    |C      |SUCCESS|16         |
+----------+------------------------------------+-----------+--------------+--------------------------+--------+-------+-------+-----------+

